# HLS Video Player — Google Colab 起動ノート

このノートブックは hls-video-player を Colab 上で動作させます。**コードベースは Google Drive から取得**、メディア（ソース動画と変換出力）も Drive に永続化します。

## 事前準備（初回のみ）

Google Drive に次のディレクトリレイアウトでコードを配置してください（ローカルの `hls-video-player/` ディレクトリをそのままアップロード）:

```
MyDrive/
└── hls-video-player/
    ├── pyproject.toml
    ├── app/
    ├── hls_video/
    ├── static/
    └── media/
        ├── source/   ← 変換元 MP4 を置く（永続）
        ├── hls/      ← 変換出力（永続、再生成可能）
        └── sprites/  ← スプライト + VTT（永続、再生成可能）
```

コード更新は手元で編集して Drive に上書き同期するだけで済みます。

## 実行手順

下の **「セットアップ & 起動」セル** を 1 回実行するだけで、FFmpeg インストール〜公開 URL 発行まで一気に行われます。

コード更新時は末尾の「コード更新時の再起動」セクションのセルを実行してください。

## セットアップ & 起動 (一括実行)

このセルを 1 回実行するだけで、FFmpeg インストールから公開 URL 発行までを一気に行います。
処理内容:

1. **FFmpeg + Python 依存** インストール、NVENC/CUVID 可用性の自己テスト
2. **Google Drive マウント** (`/content/drive`)
3. **Drive からコードベースをコピー** (`/content/hls-video-player`)、`media/` は Drive 実体への symlink
4. **Python パッケージインストール** (`pip install -e .`)
5. **環境変数設定** (`MEDIA_ROOT`, `FFMPEG_*`)
6. **FastAPI + Gradio 起動** (`*.gradio.live` の公開 URL 発行)

それぞれのステップは独立していないので、順番に実行する必要があります。
コード更新時は末尾の「コード更新時の再起動」セクションを参照してください。

In [ ]:
# ============================================================
# 1. FFmpeg + Python 依存 + LD_LIBRARY_PATH 設定
# ============================================================
!apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -1

# Colab T4 固有の NVIDIA driver パス対応:
# /usr/lib64-nvidia/ に libcuda.so.1 / libnvcuvid.so.1 が配置されているが
# ldconfig のサーチに含まれず apt ffmpeg が dlopen 失敗する問題の回避。
import os
_nvidia_lib = '/usr/lib64-nvidia'
if os.path.isdir(_nvidia_lib):
    cur = os.environ.get('LD_LIBRARY_PATH', '')
    if _nvidia_lib not in cur.split(':'):
        os.environ['LD_LIBRARY_PATH'] = _nvidia_lib + (':' + cur if cur else '')
print('LD_LIBRARY_PATH =', os.environ.get('LD_LIBRARY_PATH', ''))

!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E 'h264_nvenc|libx264' || true
!ffmpeg -hide_banner -decoders 2>/dev/null | grep -E 'h264_cuvid|hevc_cuvid|vp9_cuvid' || true
!nvidia-smi -L 2>/dev/null || echo '(GPU ランタイム未選択: CPU エンコードになります)'

print('--- CUDA device init self-test ---')
!ffmpeg -hide_banner -v error -f lavfi -i testsrc=d=0.1:s=320x240 -vf hwupload_cuda -f null - && echo 'CUVID/NVENC READY' || echo 'CUDA init FAILED (will fall back to CPU)'

!pip install -q gradio fastapi uvicorn python-multipart

# ============================================================
# 2. Google Drive マウント
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# 3. Drive からコードベースをコピー (media/ は Drive への symlink)
# ============================================================
import os, shutil

DRIVE_ROOT = '/content/drive/MyDrive/hls-video-player'
LOCAL_ROOT = '/content/hls-video-player'

assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} が見つかりません。MyDrive 直下に hls-video-player/ を配置してください。'
)

if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(
    DRIVE_ROOT,
    LOCAL_ROOT,
    ignore=shutil.ignore_patterns(
        'media', '.git', '.claude',
        '__pycache__', '.pytest_cache', '.venv', '*.egg-info',
        'node_modules',
    ),
)

drive_media = f'{DRIVE_ROOT}/media'
for sub in ('source', 'hls', 'sprites'):
    os.makedirs(f'{drive_media}/{sub}', exist_ok=True)

local_media = f'{LOCAL_ROOT}/media'
if os.path.islink(local_media) or os.path.exists(local_media):
    os.remove(local_media) if os.path.islink(local_media) else shutil.rmtree(local_media)
os.symlink(drive_media, local_media)

%cd {LOCAL_ROOT}
print(f'media symlink: {local_media} -> {os.readlink(local_media)}')

# ============================================================
# 4. Python パッケージとしてインストール
# ============================================================
!pip install -q -e .

# ============================================================
# 5. 環境変数設定 (MEDIA_ROOT + 変換パフォーマンス)
# ============================================================
os.environ['MEDIA_ROOT'] = f'{LOCAL_ROOT}/media'
os.environ['PORT'] = '7860'

# NVENC + CUVID は LD_LIBRARY_PATH を通してあるので auto で検出される
# NVENC/CUVID は LD_LIBRARY_PATH を通してあるので auto で検出される
os.environ['FFMPEG_HWACCEL'] = 'auto'              # NVENC 使えれば使う
os.environ['FFMPEG_CUVID'] = 'auto'                # CUVID 使えれば使う
os.environ['FFMPEG_NVENC_PRESET'] = 'p4'           # GPU 時のみ有効
os.environ['FFMPEG_BFRAMES'] = '0'                 # GPU 時のみ有効
# --- CPU fallback 時の最適化 (Colab 8 vCPU / 50GB RAM 向け) ---
os.environ['FFMPEG_PRESET'] = 'ultrafast'          # libx264 preset (最速)
os.environ['FFMPEG_X264_TUNE'] = 'zerolatency'     # +10-20% (画質微低下)
os.environ['FFMPEG_AUDIO_COPY'] = '1'              # 音声 AAC copy で +5%
os.environ['FFMPEG_THREADS'] = '0'                 # ffmpeg auto
os.environ['FFMPEG_VARIANTS'] = '720p,360p'        # 2 本に絞って +70%
os.environ['MAX_CONCURRENT_JOBS'] = '1'

print('MEDIA_ROOT       =', os.environ['MEDIA_ROOT'])
print('FFMPEG_HWACCEL   =', os.environ['FFMPEG_HWACCEL'])
print('FFMPEG_CUVID     =', os.environ['FFMPEG_CUVID'])
print('FFMPEG_VARIANTS  =', os.environ['FFMPEG_VARIANTS'])
print('FFMPEG_X264_TUNE =', os.environ['FFMPEG_X264_TUNE'])
print('FFMPEG_AUDIO_COPY=', os.environ['FFMPEG_AUDIO_COPY'])
!ls {os.environ['MEDIA_ROOT']}

# ============================================================
# 6. FastAPI + Gradio 起動、*.gradio.live トンネル発行
# ============================================================
# demo.launch(share=True) だと /hls/* などの静的配信が share URL で見えなく
# なるため、FastAPI 本体 (port 7860) を setup_tunnel で直接トンネル化する。
import gradio as gr
from fastapi import FastAPI

from app.gradio_ui import build_ui
from app.static_mount import mount_static
from app.player_embed import router as player_router
from hls_video.config import media_root

media = media_root()
fapi = FastAPI()
mount_static(fapi, media_root=str(media), static_dir=f'{LOCAL_ROOT}/static')
fapi.include_router(player_router)

demo = build_ui(media_dir=media)
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

import threading, time, uvicorn
_HLS_UVICORN = uvicorn.Server(uvicorn.Config(fapi, host='0.0.0.0', port=7860, log_level='warning'))
_HLS_THREAD = threading.Thread(target=_HLS_UVICORN.run, daemon=True)
_HLS_THREAD.start()
time.sleep(3)

import secrets
from gradio.networking import setup_tunnel

share_url = setup_tunnel(
    local_host='localhost',
    local_port=7860,
    share_token=secrets.token_urlsafe(32),
    share_server_address=None,
    share_server_tls_certificate=None,
)

print(f'\n🌐 公開 URL: {share_url}')
print('   /          → Gradio UI')
print('   /hls/*     → HLS プレイリスト・セグメント')
print('   /sprites/* → スプライト・WebVTT')
print('   /api, /player, /static も同じ URL 配下で利用可能')
print('\n💻 Colab 内からのアクセス: http://localhost:7860')


### 公開 URL のライフサイクル

- `*.gradio.live` のトンネルは **72 時間** で自動失効（Gradio 側の仕様）
- Colab ランタイム切断時にも失効する
- 再発行は上のセルを再実行するだけ（新しい URL が振られる）
- 長く稼働させ続けたい場合は Colab Pro + `Keep session active` や ngrok などの外部トンネルを検討

### トラブルシューティング

| 症状 | 対処 |
|---|---|
| 公開 URL を開くと 502 / 接続エラー | uvicorn が落ちた可能性。`!ss -tlnp \| grep 7860` で確認し、「セットアップ & 起動」セルを再実行 |
| `setup_tunnel` が `TunnelException` で失敗 | Gradio 共有サーバ側の障害。数分待って再実行 |
| 動画再生は始まるがシーク/サムネが出ない | `/sprites/<id>.jpg` が 200 を返すか公開 URL 経由で curl で確認 |
| アップロードや変換で 413 / タイムアウト | Drive I/O 負荷。`MAX_CONCURRENT_JOBS=1` を維持、大ファイルは分割して処理 |

## ログ確認 (リアルタイム)

変換や probe の進捗は別スレッドで動くため、「セットアップ & 起動」セルの実行が終わった後は
そのセル出力には追加のログが流れない（Colab 仕様）。
`setup_logging()` は `/tmp/hls-video.log` にもファイル出力しているので、
別セルで `!tail -f` すれば進捗・警告・エラーがリアルタイムに読める。

- 直近 100 行を眺めるだけ: 下の「ログ末尾」セル
- リアルタイムで追う: 下の「ログをストリーム表示」セル（停止はセルの停止ボタン）

In [ ]:
# ログ末尾（一発実行）
!tail -n 200 /tmp/hls-video.log 2>/dev/null || echo '(ログファイルがまだありません。セル 6 を実行したか確認してください)'

In [ ]:
# ログをストリーム表示（止めるにはセルの停止ボタン ⬛）
!touch /tmp/hls-video.log && tail -n 50 -F /tmp/hls-video.log

## 7. コード更新時の再起動

手元で編集 → `./scripts/sync-to-drive.sh` で Drive に同期したあと、
このセクションのセルを 1 つ実行するだけでアプリを最新コードで再起動できる。

処理内容:
1. 既存の uvicorn サーバを停止（`_HLS_UVICORN.should_exit = True`）
2. `frpc` トンネルプロセスを終了（新しい `*.gradio.live` URL を発行するため）
3. Drive から `LOCAL_ROOT` へコード再コピー、`media/` symlink を作り直し
4. `pip install -e .`（`pyproject.toml` 変更時だけ実質何か起きる）
5. `sys.modules` から `app.*` / `hls_video.*` を破棄して再読込
6. FastAPI + Gradio を再起動、新しい `*.gradio.live` URL を表示

※ Drive マウントや FFmpeg 等の環境構築セルは触らない。Python kernel が
   生きていれば 10 秒ほどで再起動できる。

In [ ]:
# --- LD_LIBRARY_PATH の再確認 (カーネル再接続対策) ---
import os
if '/usr/lib64-nvidia' not in os.environ.get('LD_LIBRARY_PATH', '').split(':'):
    os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')

# 1) 既存サーバ停止
try:
    _HLS_UVICORN.should_exit = True
    _HLS_THREAD.join(timeout=5)
    print('✓ uvicorn stopped')
except NameError:
    print('(前回起動なし)')
except Exception as e:
    print(f'stop error: {e}')

import subprocess, time
# Gradio の共有トンネル (frpc サブプロセス) を止める
subprocess.run(['pkill', '-f', 'frpc'], check=False)
time.sleep(1)

# 2) Drive からコード再コピー + media symlink 作り直し
import os, shutil
if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(
    DRIVE_ROOT, LOCAL_ROOT,
    ignore=shutil.ignore_patterns(
        'media', '.git', '.claude',
        '__pycache__', '.pytest_cache', '.venv', '*.egg-info',
        'node_modules',
    ),
)
drive_media = f'{DRIVE_ROOT}/media'
for sub in ('source', 'hls', 'sprites'):
    os.makedirs(f'{drive_media}/{sub}', exist_ok=True)
os.symlink(drive_media, f'{LOCAL_ROOT}/media')
%cd {LOCAL_ROOT}
print('✓ code re-copied')

# 3) 依存関係更新（pyproject 変更時のため）
!pip install -q -e .

# 4) 既にロード済みのモジュールを sys.modules から破棄 → 次の import で最新が読まれる
import sys
purged = [m for m in list(sys.modules) if m == 'app' or m == 'hls_video' or m.startswith(('app.', 'hls_video.'))]
for m in purged:
    sys.modules.pop(m, None)
print(f'✓ purged {len(purged)} modules')

# 5) FastAPI + Gradio を再起動（セル 6 の内容を再実行）
import gradio as gr
from fastapi import FastAPI
from app.gradio_ui import build_ui
from app.static_mount import mount_static
from app.player_embed import router as player_router
from hls_video.config import media_root

media = media_root()
fapi = FastAPI()
mount_static(fapi, media_root=str(media), static_dir=f'{LOCAL_ROOT}/static')
fapi.include_router(player_router)

demo = build_ui(media_dir=media)
demo.queue()
gr.mount_gradio_app(fapi, demo, path='/')

import threading, uvicorn
_HLS_UVICORN = uvicorn.Server(uvicorn.Config(fapi, host='0.0.0.0', port=7860, log_level='warning'))
_HLS_THREAD = threading.Thread(target=_HLS_UVICORN.run, daemon=True)
_HLS_THREAD.start()
time.sleep(3)

# 6) 新しい Gradio 共有トンネルを発行
import secrets
from gradio.networking import setup_tunnel
share_url = setup_tunnel(
    local_host='localhost', local_port=7860,
    share_token=secrets.token_urlsafe(32),
    share_server_address=None, share_server_tls_certificate=None,
)
print(f'\n🔄 再起動完了')
print(f'🌐 新しい公開 URL: {share_url}')